# Autogen Multiple Agents team interaction

In [2]:
# Importing libraries
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_ext.tools.langchain import LangChainToolAdapter
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.models.ollama import OllamaChatCompletionClient
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [17]:
# Adding the serper tool
serper = GoogleSerperAPIWrapper()
lang_serper = Tool(name="Internet_search", func=serper.run, description="userful tool for searching the web")
autogen_serper = LangChainToolAdapter(lang_serper)

# Definiting model
openai_model = OpenAIChatCompletionClient(model="gpt-4o-mini")

# Assigning prompt
prompt =""" Find the best middle eastern receipes for healthy eating."""

# Defining agents
worker_agent = AssistantAgent(
    name="worker_agent",
    model_client= openai_model,
    tools=[autogen_serper],
    system_message="You are a helpful resarch assistant who looks for good restuarants and foods, consider any feedback you receive.",
    reflect_on_tool_use=True
)

evaluator_agent = AssistantAgent(
    name="evaluator_agent",
    model_client=openai_model,
    system_message="Provide constructive feedback. Respond with 'SATISFIED' when the user requirement is complete."
)

termination_text = TextMentionTermination("SATISFIED")

# Defining the team of agents
team = RoundRobinGroupChat(participants=[worker_agent, evaluator_agent], termination_condition=termination_text, max_turns=15)

In [15]:
# Calling the team
response = await team.run(task = prompt)
for message in response.messages:
    print(f"{message.source}:{message.content}\n\n")

user: Find the best middle eastern receipes for healthy eating.


worker_agent:[FunctionCall(id='call_djfxqqyfCJEJ8E3Uqr5k8Ocy', arguments='{"query":"best healthy Middle Eastern recipes"}', name='Internet_search')]


worker_agent:[FunctionExecutionResult(content="Fresh, bold, and flavorful, our favorites include Shakshuka, Authentic Falafels, our ever-popular Chicken Shawarma Recipe, and Lemony Lebanese Tabouli. Discover my collection of healthy Middle Eastern recipes inspired by the authentic and fusion cuisines, including Turkish, Lebanese, Syrian, and Israeli dishes. Healthy Arabic recipes for dinner crafted by a registered dietitian. Learn how to make Middle Eastern recipes with a healthy twist! Many of these dishes are naturally plant-based, making vegan Middle Eastern food easy to enjoy. Syrian foul, a fava bean breakfast salad, and harak osbao. For protein chickpeas, lentils and favs beans are staples. Chickpeas you can put them in salads, with hummus, or muhamara (best dip in t

### Let's change the prompt and try

In [18]:
# Assigning prompt
prompt =""" Find the best middle eastern receipes for healthy eating.Suggest top 3 recipes."""

# Calling the team
response = await team.run(task = prompt)
for message in response.messages:
    print(f"{message.source}:{message.content}\n\n")


user: Find the best middle eastern receipes for healthy eating.Suggest top 3 recipes.


worker_agent:[FunctionCall(id='call_pLm1sl5qe4vXaZkkUFvvNsbT', arguments='{"query":"best healthy Middle Eastern recipes"}', name='Internet_search')]


worker_agent:[FunctionExecutionResult(content="Fresh, bold, and flavorful, our favorites include Shakshuka, Authentic Falafels, our ever-popular Chicken Shawarma Recipe, and Lemony Lebanese Tabouli. Find healthy, delicious middle Eastern recipes including Lebanese, Israeli and Turkish recipes. Healthier Recipes, from the food and nutrition experts at ... Discover my collection of healthy Middle Eastern recipes inspired by the authentic and fusion cuisines, including Turkish, Lebanese, Syrian, and Israeli dishes. Healthy Arabic recipes for dinner crafted by a registered dietitian. Learn how to make Middle Eastern recipes with a healthy twist! For protein chickpeas, lentils and favs beans are staples. Chickpeas you can put them in salads, with hummus, o